In [1]:
import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

c:\Users\Kelsey\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
hidden_df = pd.read_csv("hidden_test_with_labels.csv")

print("Hidden Test Shape:", hidden_df.shape)
hidden_df.head()

Hidden Test Shape: (600, 5)


,id,text,label,label_name,source_file
0,neg_cv795_10291,"mr . bean , a bumbling security guard from eng...",0,negative,neg/cv795_10291.txt
1,neg_cv174_9735,"starship troopers is a bad movie . \ni mean , ...",0,negative,neg/cv174_9735.txt
2,pos_cv065_15248,"what a great film . \nwhat a stunning , touchi...",1,positive,pos/cv065_15248.txt
3,neg_cv076_26009,"susan granger's review of "" the watcher "" ( un...",0,negative,neg/cv076_26009.txt
4,neg_cv417_14653,the marvelous british actor derek jacobi stars...,0,negative,neg/cv417_14653.txt


In [3]:
checkpoint_path = "model_checkpoint"

tokenizer = DistilBertTokenizerFast.from_pretrained(checkpoint_path)

model = DistilBertForSequenceClassification.from_pretrained(
    checkpoint_path
)

model.to(device)
model.eval()

print("Stage 1 model loaded successfully.")


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2160.80it/s]

Stage 1 model loaded successfully.


In [4]:
class HiddenReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }


hidden_dataset = HiddenReviewDataset(
    hidden_df["text"],
    tokenizer,
    max_length=256
)

hidden_loader = DataLoader(
    hidden_dataset,
    batch_size=8,
    shuffle=False
)

print("Hidden examples:", len(hidden_dataset))
print("Number of batches:", len(hidden_loader))

Hidden examples: 600
Number of batches: 75


In [5]:
hidden_predictions = []

model.eval()

with torch.no_grad():
    for batch in hidden_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        hidden_predictions.extend(
            predictions.cpu().numpy()
        )

print("Predictions generated:", len(hidden_predictions))

Predictions generated: 600


In [6]:
hidden_true_labels = hidden_df["label"].tolist()

hidden_accuracy = accuracy_score(
    hidden_true_labels,
    hidden_predictions
)

hidden_cm = confusion_matrix(
    hidden_true_labels,
    hidden_predictions
)

print("Hidden Test Accuracy:", hidden_accuracy)
print("\nConfusion Matrix:")
print(hidden_cm)

print("\nClassification Report:")
print(
    classification_report(
        hidden_true_labels,
        hidden_predictions,
        target_names=["Negative", "Positive"]
    )
)

Hidden Test Accuracy: 0.5

Confusion Matrix:
[[273  27]
 [273  27]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.50      0.91      0.65       300
    Positive       0.50      0.09      0.15       300

    accuracy                           0.50       600
   macro avg       0.50      0.50      0.40       600
weighted avg       0.50      0.50      0.40       600



In [7]:
hidden_predictions_df = pd.DataFrame({
    "id": hidden_df["id"],
    "predicted_label": hidden_predictions
})

hidden_predictions_df.to_csv(
    "hidden_test_predictions.csv",
    index=False
)

print("hidden_test_predictions.csv saved successfully.")
hidden_predictions_df.head()

hidden_test_predictions.csv saved successfully.


,id,predicted_label
0,neg_cv795_10291,0
1,neg_cv174_9735,1
2,pos_cv065_15248,0
3,neg_cv076_26009,0
4,neg_cv417_14653,0


The model got 50% accuracy on the hidden test set. The confusion matrix was [[273, 27], [273, 27]]. The model correctly predicted 273 out of 300 negative reviews, but it only correctly predicted 27 out of 300 positive reviews. This shows that the model was much better at predicting negative reviews and had trouble identifying positive reviews. On the public test set, the model had 52.5% accuracy, while on the hidden test set it had 50% accuracy. The results were pretty close, with only a 2.5% difference. The hidden test results also showed that the model was predicting negative reviews much more often than positive reviews. One reason for this could be the small and imbalanced training set, which made it harder for the model to perform well on new reviews. If I could apply more, I would try different learning rates, batch sizes, and numbers of epochs to see if the model could perform better. I would also try another way of handling the class imbalance and compare the results with another pretrained model. Since the training set was small, I would try to improve the model while also making sure it does not overfit the output im looking for.